### Imports

In [1]:
import tensorflow as tf
import numpy as np
import cv2 # For high-quality resizing
import os
import json
from PIL import Image
import glob
import sys
from tqdm import tqdm

sys.path.insert(0, "../../")
from config import MEDIA_PATH, CROPPED_PATH, MODELS_PATH, IMG_SIZE

2025-09-27 15:22:24.984305: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-27 15:22:25.008553: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-27 15:22:25.655525: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Functions

In [2]:
def define_prompt(path):
    PROPHASE_PROMPT = "allium cepa root tip cell in prophase state"
    METAPHASE_PROMPT = "allium cepa root tip cell in metaphase state"
    ANAPHASE_PROMPT = "allium cepa root tip cell in anaphase state"
    TELOPHASE_PROMPT = "allium cepa root tip cell in telophase state"
    if 'prophase' in path:
        return PROPHASE_PROMPT
    elif 'metaphase' in path:
        return METAPHASE_PROMPT
    elif 'anaphase' in path:
        return ANAPHASE_PROMPT
    elif 'telophase' in path:
        return TELOPHASE_PROMPT

In [3]:
class Sampling(tf.keras.layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [4]:
def load_vae_models(encoder_path, decoder_path):
    """Loads the pre-trained Keras VAE encoder and decoder models."""
    print("Loading VAE models...")
    # We must tell Keras about our custom 'Sampling' layer if the encoder uses it
    custom_objects = {"Sampling": Sampling}
    encoder = tf.keras.models.load_model(encoder_path, custom_objects=custom_objects)
    decoder = tf.keras.models.load_model(decoder_path)
    print("VAE models loaded successfully.")
    return encoder, decoder


In [5]:
def preprocess_image_for_vae(image_path, target_size):
    """Loads and preprocesses a single image for VAE input."""
    img_raw = tf.io.read_file(image_path)
    img = tf.image.decode_image(img_raw, channels=1, expand_animations=False) # Decode as 1-channel grayscale
    img_resized = tf.image.resize(img, target_size)
    img_normalized = tf.cast(img_resized, tf.float32) / 255.0
    return tf.expand_dims(img_normalized, axis=0) # Add batch dimension

In [6]:
def generate_vae_reconstruction(encoder_model, decoder_model, input_batch):
    """Runs the encoder-decoder pipeline to get the VAE reconstruction."""
    z_mean, z_log_var, z_sampled = encoder_model.predict(input_batch, verbose=0)
    reconstructed_batch = decoder_model.predict(z_sampled, verbose=0)
    return reconstructed_batch[0] # Get the single image from batch

In [7]:
def resize_image_cv2(image_array, target_size, interpolation=cv2.INTER_LANCZOS4):
    """Resizes a grayscale image using OpenCV for better quality upscaling."""
    # Ensure image_array is 2D for grayscale processing, then add back channel
    if image_array.ndim == 3 and image_array.shape[-1] == 1:
        image_array = image_array.squeeze(-1) # Remove channel dim for cv2 resize
    resized_img = cv2.resize(image_array, target_size, interpolation=interpolation)
    return np.expand_dims(resized_img, axis=-1) # Add channel dim back


In [8]:
def save_image_grayscale(image_array, output_path):
    """Saves a grayscale (0-1 float) NumPy image array to a file."""
    # Convert float [0,1] to uint8 [0,255]
    img_uint8 = (image_array.squeeze() * 255).astype(np.uint8)
    Image.fromarray(img_uint8, mode='L').save(output_path)

In [9]:
def create_controlnet_dataset(
    all_original_image_paths,
    encoder_model,
    decoder_model,
    vae_input_size,
    target_resolution_controlnet,
    controlnet_dataset_root,
    generic_prompt
):
    """
    Main function to generate VAE reconstructions, upscale, and prepare the ControlNet dataset.
    """
    print(f"\n--- Starting Data Generation for ControlNet (Total Images: {len(all_original_image_paths)}) ---")

    # Create output directories
    sharp_upscaled_dir = os.path.join(controlnet_dataset_root, "sharp_upscaled")
    blurred_upscaled_dir = os.path.join(controlnet_dataset_root, "blurred_upscaled")
    os.makedirs(sharp_upscaled_dir, exist_ok=True)
    os.makedirs(blurred_upscaled_dir, exist_ok=True)

    metadata_list = []

    for i, original_image_path in enumerate(tqdm(all_original_image_paths, desc="Processing images")):
        # print(f"Processing image {i+1}/{len(all_original_image_paths)}: {original_image_path}")

        # --- 1. Load and Preprocess Original Image for VAE ---
        input_batch = preprocess_image_for_vae(original_image_path, vae_input_size)
        original_img_normalized_np = input_batch[0].numpy() # Get the 0-1 normalized numpy array

        # --- 2. Generate Blurred Image from VAE ---
        reconstructed_image_np = generate_vae_reconstruction(encoder_model, decoder_model, input_batch)

        # --- 3. Upscale Both Images to Target Resolution ---
        base_name = os.path.splitext(os.path.basename(original_image_path))[0]
        sharp_upscaled_filename = f"{base_name}_sharp_{target_resolution_controlnet}x{target_resolution_controlnet}.png"
        blurred_upscaled_filename = f"{base_name}_blurred_{target_resolution_controlnet}x{target_resolution_controlnet}.png"

        sharp_upscaled_path_full = os.path.join(sharp_upscaled_dir, sharp_upscaled_filename)
        blurred_upscaled_path_full = os.path.join(blurred_upscaled_dir, blurred_upscaled_filename)

        # Upscale original sharp image
        upscaled_sharp_np = resize_image_cv2(original_img_normalized_np,
                                              (target_resolution_controlnet, target_resolution_controlnet),
                                              interpolation=cv2.INTER_LANCZOS4)
        save_image_grayscale(upscaled_sharp_np, sharp_upscaled_path_full)

        # Upscale VAE blurred image
        upscaled_blurred_np = resize_image_cv2(reconstructed_image_np,
                                               (target_resolution_controlnet, target_resolution_controlnet),
                                               interpolation=cv2.INTER_LANCZOS4)
        save_image_grayscale(upscaled_blurred_np, blurred_upscaled_path_full)


        # --- 4. Add to Metadata ---
        metadata_list.append({
            "file_name": os.path.join("sharp_upscaled", sharp_upscaled_filename),
            "conditioning_image": os.path.join("blurred_upscaled", blurred_upscaled_filename),
            "text": define_prompt(original_image_path) or generic_prompt
        })

    # Save metadata.jsonl
    metadata_path = os.path.join(controlnet_dataset_root, "metadata.jsonl")
    with open(metadata_path, "w") as f:
        for item in metadata_list:
            f.write(json.dumps(item) + "\n")

    print(f"\n--- Data generation complete. Metadata saved to {metadata_path} ---")
    print(f"ControlNet dataset prepared in: {controlnet_dataset_root}")

### Global definitions

In [10]:
TARGET_RESOLUTION_CONTROLNET = 512 # Target resolution for ControlNet training
GENERIC_PROMPT = "a micrograph of an allium cepa root tip mitotic cell"
VAE_ENCODER_PATH = os.path.join(MODELS_PATH, 'vae', 'm1_encoder_mitosis_canny.keras')
VAE_DECODER_PATH = os.path.join(MODELS_PATH, 'vae', 'm1_decoder_mitosis_canny.keras')
ORIGINAL_IMAGES_DIR = "original_sharp_cells" # Directory containing your original images

### Creation of the train dataset

In [11]:
# --- Configuration ---
# Update these paths to match your project structure
CONTROLNET_DATASET_ROOT = os.path.join(CROPPED_PATH, 'controlnet_dataset', 'train')

IMAGES_UNLABELED_PATH = os.path.join(CROPPED_PATH, 'vae', 'train', 'untagged')
IMAGES_LABELED_PATH = os.path.join(CROPPED_PATH, 'vae', 'train', 'tagged')

unlabeled_files = [f for f in glob.glob(os.path.join(IMAGES_UNLABELED_PATH, '*')) if "aug" not in os.path.basename(f)] #I do not include augmented images since they introduce noise to the difussor
labeled_files = [f for f in glob.glob(os.path.join(IMAGES_LABELED_PATH, '*/*')) if "aug" not in os.path.basename(f)]
all_original_image_paths = unlabeled_files + labeled_files


if not all_original_image_paths:
    print(f"ERROR: No images found in {ORIGINAL_IMAGES_DIR} or your image list is empty.")
    print("Please ensure your 'all_original_image_paths' list is correctly populated.")
else:
    # Load VAE models
    # Ensure the VAE_ENCODER_PATH and VAE_DECODER_PATH are correct relative to where you run this script.
    encoder, decoder = load_vae_models(VAE_ENCODER_PATH, VAE_DECODER_PATH)

    # Run the dataset creation
    create_controlnet_dataset(
        all_original_image_paths,
        encoder,
        decoder,
        IMG_SIZE,
        TARGET_RESOLUTION_CONTROLNET,
        CONTROLNET_DATASET_ROOT,
        GENERIC_PROMPT
    )

Loading VAE models...


I0000 00:00:1758997349.285692 1158262 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21251 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:01:00.0, compute capability: 8.6


VAE models loaded successfully.

--- Starting Data Generation for ControlNet (Total Images: 4064) ---


Processing images:   0%|          | 0/4064 [00:00<?, ?it/s]2025-09-27 15:22:30.013113: I external/local_xla/xla/service/service.cc:163] XLA service 0x79667c0036e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-27 15:22:30.013124: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2025-09-27 15:22:30.026111: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-27 15:22:30.056397: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90701
I0000 00:00:1758997350.489011 1158516 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
Processing images: 100%|██████████| 4064/4064 [06:57<00:00,  9.73it/s]


--- Data generation complete. Metadata saved to /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/controlnet_dataset/train/metadata.jsonl ---
ControlNet dataset prepared in: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/controlnet_dataset/train


### Creation of the test dataset

In [ ]:
# --- Configuration ---
# Update these paths to match your project structure
CONTROLNET_DATASET_ROOT = os.path.join(CROPPED_PATH, 'controlnet_dataset', 'test')
ORIGINAL_IMAGES_DIR = "original_sharp_cells" # Directory containing your original images

IMAGES_LABELED_PATH = os.path.join(CROPPED_PATH, 'vae', 'test')

all_original_image_paths = glob.glob(os.path.join(IMAGES_LABELED_PATH, '*/*'))

if not all_original_image_paths:
    print(f"ERROR: No images found in {ORIGINAL_IMAGES_DIR} or your image list is empty.")
    print("Please ensure your 'all_original_image_paths' list is correctly populated.")
else:
    # Load VAE models
    # Ensure the VAE_ENCODER_PATH and VAE_DECODER_PATH are correct relative to where you run this script.
    encoder, decoder = load_vae_models(VAE_ENCODER_PATH, VAE_DECODER_PATH)

    # Run the dataset creation
    create_controlnet_dataset(
        all_original_image_paths,
        encoder,
        decoder,
        IMG_SIZE,
        TARGET_RESOLUTION_CONTROLNET,
        CONTROLNET_DATASET_ROOT,
        GENERIC_PROMPT
    )

Loading VAE models...
VAE models loaded successfully.

--- Starting Data Generation for ControlNet (Total Images: 242) ---


Processing images: 100%|██████████| 242/242 [00:24<00:00,  9.87it/s]


--- Data generation complete. Metadata saved to /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/controlnet_dataset/test/metadata.jsonl ---
ControlNet dataset prepared in: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/controlnet_dataset/test
